# Evaluación de los métodos

## Carga del dataset y calculo de atributos


In [1]:
import pandas as pd
from load import load_dataset

dataset = load_dataset("futbol_uruguayo.csv")

print(dataset.head())


                   home                              away       date  gh  ga  \
0           Bella Vista                 Defensor Sporting 1932-03-05   1   2   
1            CA Penarol                       River Plate 1932-03-05   1   1   
2       Central Espanol        Rampla Juniors Futbol Club 1932-03-05   1   0   
3  Montevideo Wanderers                       Racing Club 1932-03-05   3   0   
4              Nacional  Institucion Atletica Sud America 1932-03-05   2   0   

  result  record_difference  last_matches_difference  goal_difference_value  \
0      V                0.0                      0.0                    0.0   
1      E                0.0                      0.0                    0.0   
2      L                0.0                      0.0                    0.0   
3      L                0.0                      0.0                    0.0   
4      L                0.0                      0.0                    0.0   

   attack_difference  defense_difference  dr

## División del conjunto en entrenamiento y evaluación

In [ ]:
#separamos cronologicamente el conjunto de entrenamiento y el de evaluacion
#la evaluacion se mantiene separada hasta haber elegido los hiperparametros

train = dataset[
    dataset["date"] < pd.Timestamp("2024-01-01")
].copy()

test = dataset[
    (dataset["date"] >= pd.Timestamp("2024-01-01"))
    & (dataset["date"] < pd.Timestamp("2026-01-01"))
].copy()

print("Cantidad de partidos de entrenamiento:", len(train))
print("Cantidad de partidos de evaluacion:", len(test))

## Creación del pipeline

In [ ]:
import sys
import os

from pipeline import create_model_pipeline, pipeline_input_attributes


# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))


#hacemos el arbol de decision usando solamente el conjunto de entrenamiento
from decisionTree.classifier import Classifier as DecisionTreeClassifier

treeID3 = DecisionTreeClassifier(min_info_gain=0.01)

model = create_model_pipeline(treeID3)

In [ ]:
X_train = train[pipeline_input_attributes].copy()
y_train = train["result"].copy()

X_test = test[pipeline_input_attributes].copy()
y_test = test["result"].copy()

print("Filas de entrenamiento:", len(X_train))
print("Filas reservadas para evaluacion final:", len(X_test))

## División temporal por temporada para Cross-Validation

Se hizo esto por ... TODO: Justificar 

In [ ]:
import numpy as np

#cada temporada se valida usando solamente las temporadas anteriores
validation_years = [2020, 2021, 2022, 2023]
train_years = train["date"].dt.year.to_numpy()
temporal_splits = []

for validation_year in validation_years:
    fit_indices = np.flatnonzero(
        train_years < validation_year
    )
    validation_indices = np.flatnonzero(
        train_years == validation_year
    )

    temporal_splits.append((
        fit_indices,
        validation_indices
    ))

    print(
        f"Validacion {validation_year}:",
        f"entrenamiento={len(fit_indices)},",
        f"validacion={len(validation_indices)}"
    )

In [ ]:
from sklearn.model_selection import GridSearchCV

#en esta primera busqueda variamos solamente el margen de record
param_grid = {
    #diferencia de tasa de victorias
    "preprocessing__differences__discretizer__record_margin": [
        0.00,
        0.025,
        0.05,
    ],

    #diferencia de forma reciente
    "preprocessing__differences__discretizer__last_matches_margin": [
        0.00,
        0.07,
        0.14,
    ],

    #diferencia de goles general
    "preprocessing__differences__discretizer__goal_difference_margin": [
        0.00,
        0.25,
        0.50,
    ],

    #diferencia de goles convertidos
    "preprocessing__differences__discretizer__attack_margin": [
        0.00,
        0.25,
        0.50,
    ],

    #diferencia de goles recibidos
    "preprocessing__differences__discretizer__defense_margin": [
        0.00,
        0.25,
        0.50,
    ],

    #ganancia minima del ID3
    "model__min_info_gain": [
        0.0000,
        0.0025,
        0.0050,
        0.0075,
        0.0100,
        0.0200,
    ],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=temporal_splits,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Mejores hiperparametros:")
print(grid_search.best_params_)
print(f"Mejor accuracy temporal: {grid_search.best_score_:.3%}")

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)

results = results[
    [
        "param_preprocessing__differences__discretizer__record_margin",
        "param_model__min_info_gain",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("rank_test_score")

results.head(10)